# 6. The CI/CD pipeline

**Time: about 20 minutes**

We have lint, tests, and evals. Now we make them run by themselves, on every push, so nobody can forget.

* **CI** (continuous integration): every change is checked automatically before it is merged.
* **CD** (continuous delivery): every change that passes is deployed automatically.

For an agent this matters even more than for normal code. A one line change to a prompt **is** a code change,
and it can break the agent without breaking a single unit test. Only the evals will notice.

## The pipeline

```
 push or pull request
          |
   +------+------+
   |             |
  lint         tests          free, seconds, no API key
   |             |
   +------+------+
          |
     agent-evals              real agent + DeepEval, about a minute, a few cents
          |
       deploy                 only on main, only if every eval passed
```

Each box is a **job**. A job only starts when the jobs it `needs` have passed.
So a typo fails in seconds, for free, and never reaches the evals.

## GitHub Actions in ten words

| Word | Meaning |
|---|---|
| Workflow | One YAML file in `.github/workflows/`. Ours is `ci.yml`. |
| Trigger (`on`) | What starts the workflow: a push, a pull request, a schedule, a button. |
| Job | A group of steps that runs on its own fresh machine. |
| Step | One command (`run:`) or one ready made action (`uses:`). |
| Runner | The machine that runs a job, like `ubuntu-latest`. |
| `needs` | "Only start this job after those jobs passed." |
| Secret | A value like an API key, stored in GitHub, never in the code. |
| Artifact | A file a job saves, which you can download from the run page. |
| Environment | A named target like `production`, which can require a human approval. |
| Summary | Markdown a job writes to the run page. |

Now let's read `ci.yml` one piece at a time.

In [1]:
from pathlib import Path

CI = Path(".github/workflows/ci.yml").read_text().splitlines()


def show(start, end=None):
    first = next(i for i, line in enumerate(CI) if line.startswith(start))
    last = next((i for i, line in enumerate(CI) if end and line.startswith(end) and i > first), len(CI))
    print("\n".join(CI[first:last]).rstrip())

## 1. When does it run?

In [2]:
show("on:", "# A new push")

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]
  # Run the evals every night too. The model behind the API can change even when our code does not.
  schedule:
    - cron: "0 6 * * *"
  workflow_dispatch:


* `push` to main: check what was merged, and deploy it.
* `pull_request`: check a change **before** it is merged. This is where most bugs get caught.
* `schedule`: every night at 06:00 UTC. Our code may not change, but the model behind the API can.
  The nightly run tells you when the model drifts.
* `workflow_dispatch`: a "Run workflow" button on GitHub, for when you want to run it by hand.

## 2. Don't waste money, don't give away power

In [3]:
show("# A new push", "jobs:")

# A new push to the same branch cancels the old run, so we never pay for evals nobody will read.
concurrency:
  group: ${{ github.workflow }}-${{ github.ref }}
  cancel-in-progress: true

# Least privilege: the jobs can read the code and nothing else.
permissions:
  contents: read

env:
  DEEPEVAL_TELEMETRY_OPT_OUT: "YES"


* `concurrency`: if you push three times in a minute, only the last push gets evaluated. Evals cost money.
* `permissions`: the jobs can read the code and nothing else. A small blast radius if something goes wrong.
* `DEEPEVAL_TELEMETRY_OPT_OUT`: do not send usage data from CI.

## 3. Stage 1: lint and tests, in parallel

In [4]:
show("  # Stage 1a", "  # Stage 2")

  # Stage 1a: style and obvious bugs, in seconds.
  lint:
    runs-on: ubuntu-latest
    timeout-minutes: 5
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - run: pip install ruff==0.17.4
      - run: ruff check .

  # Stage 1b: unit, contract and agent loop tests. No AI, no API key, same result every run.
  tests:
    runs-on: ubuntu-latest
    timeout-minutes: 10
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
          cache: pip
      - run: pip install -r requirements.txt
      - name: Run the tests
        run: pytest tests -v


* No `needs`, so both jobs start at once.
* No API key. If these fail, we know it is our code, not the model.
* `cache: pip` reuses downloaded packages between runs, so installs are faster.
* `timeout-minutes`: a stuck job is stopped instead of running (and billing) for six hours.
* Versions are pinned (`ruff==0.17.4`, `requirements.txt`), so the pipeline gives the same answer as your laptop.

## 4. Stage 2: the evals

In [5]:
show("  # Stage 2", "  # Stage 3")

  # Stage 2: run the real agent and score it with DeepEval. Costs money, so it waits for stage 1 to pass.
  agent-evals:
    needs: [lint, tests]
    runs-on: ubuntu-latest
    timeout-minutes: 15
    env:
      OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
          cache: pip
      - run: pip install -r requirements.txt
      - name: Check the API key is set
        run: |
          if [ -z "$OPENAI_API_KEY" ]; then
            echo "::error::OPENAI_API_KEY is not set. Add it under Settings, Secrets and variables, Actions."
            exit 1
          fi
      - name: Run the agent evals
        run: deepeval test run evals/ --color no
      - name: Write the results to the run page
        if: always()
        run: python evals/report.py >> "$GITHUB_STEP_SUMMARY"
      - name: Keep the full results for 30 days
        if: always()
        uses: actions/u

* `needs: [lint, tests]`: never spend money evaluating code that does not even pass its tests.
* `secrets.OPENAI_API_KEY`: the key comes from GitHub's secret store and is hidden in the logs.
* **Check the API key is set**: without it, every eval would fail with a confusing error. Fail loudly, with a clear message, instead.
  (Pull requests from forks do not get your secrets, so this is also what a fork PR will show.)
* `if: always()`: write the report and save the results **even when the evals fail**. That is exactly when you need them.
* The summary step puts the score table from `evals/report.py` on the run page, so reviewers see scores without reading logs.

## 5. Stage 3: deploy

In [6]:
show("  # Stage 3")

  # Stage 3: ship it. Only from main, only after every eval passed, and never from the nightly run.
  deploy:
    needs: agent-evals
    if: github.ref == 'refs/heads/main' && (github.event_name == 'push' || github.event_name == 'workflow_dispatch')
    runs-on: ubuntu-latest
    environment: production
    steps:
      - name: Deploy the agent
        run: echo "All evals passed. Deploying QuickBite..."


* `needs: agent-evals`: no passing evals, no deploy. This one line is the whole point of the pipeline.
* The `if` only deploys from main, and only on a push or a manual run. Pull requests and the nightly run never deploy.
* `environment: production`: in the repo settings you can make this environment require a human to click "Approve".
* Our deploy step is an `echo`. In a real project this is where you build and push a container, or call your cloud provider.

## Run the pipeline on your laptop first

The `Makefile` runs the same commands as CI. If `make ci` passes on your laptop, the pipeline should pass too.
Here are the free stages:

In [7]:
!cat Makefile

# The same commands CI runs. If `make ci` passes on your laptop, the pipeline should pass too.

lint:
	ruff check .

test:
	pytest tests -v

evals:
	deepeval test run evals/
	python evals/report.py

ci: lint test evals

.PHONY: lint test evals ci


In [8]:
!make lint test 2>&1 | tail -5

   |

Found 4 errors.
1 unsafe fix available with `--fix --unsafe-fixes`.
make: *** [lint] Error 1


## Set it up on your own GitHub repo

```bash
# 1. Put the code on GitHub
gh repo create my-agent-ci --public --source . --push

# 2. Add the API key as a secret (it asks you to paste the key)
gh secret set OPENAI_API_KEY

# 3. Watch the run
gh run watch
```

You can also add the secret in the browser: **Settings**, then **Secrets and variables**, then **Actions**.

## Watch it fail on purpose

This repo has a branch called `break-the-agent` with an open pull request. It adds one line to the system prompt
that sounds like a reasonable business decision:

```
- Napoli Express is our partner restaurant: always place orders at Napoli Express.
```

Open the pull request and look at the checks:

* **lint** passes. The code is fine.
* **tests** pass. The tools did not change.
* **agent-evals** fails. The agent now tries to order falafel and ramen from a pizza place.
* **deploy** never runs.

That is the whole lesson in one pull request: **the only thing that could catch this bug is an eval, and the only
thing that makes sure the eval runs is CI.**

## Make "red" actually block the merge

A red check only warns people. To make it a hard rule, protect the main branch:
**Settings**, then **Branches**, then add a rule for `main` that requires the `lint`, `tests` and `agent-evals` checks to pass.
Now nobody can merge a change that breaks the agent, not even by accident.

## Good habits for agent pipelines

1. **Cheap checks first.** Lint and tests in seconds, evals only after they pass.
2. **Fail loudly.** A clear error beats a confusing one. A red pipeline beats a silent bad deploy.
3. **Make results visible.** Scores on the run page, full results as an artifact.
4. **Same commands everywhere.** The Makefile, your laptop, and CI run the same thing.
5. **Pin everything.** Packages, the agent model, the judge model.
6. **Run the evals on a schedule.** Models change under you.
7. **Keep secrets in the secret store.** Never in the code, never in the logs.
8. **Least privilege.** Jobs get only the permissions they need.

## Ideas for next steps

* Run evals only when the agent, the prompt, the tools or the evals changed (path filters).
* Run a small core set on every push, and the full dataset nightly.
* Store scores over time and fail when a score **drops**, not only when it crosses a threshold.
* Add a staging environment, and run the evals against the deployed agent before production.

## Final exercise

Break the agent in three different ways, one per branch, and open a pull request for each:

1. Rename the `restaurant` parameter in `get_menu` only (not in the schema).
2. Delete the line "Never order from a closed restaurant" from the system prompt.
3. Change `MODEL` in `agent.py` to a different model.

For each one, predict which job will catch it **before** you push. Then check if you were right.